In [1]:
from functools import wraps
from typing import Callable, Any
from datetime import datetime as dt
import time

## Positional vs. Keyword arguments

In [2]:
def fun(a, b, c):
    return a + b - c

In [3]:
# Specify arguments positionally
fun(1, 2, 3)

0

In [4]:
# Keyword specification
fun(b=2, c=3, a=1)

0

In [5]:
# Do a mixture, positional has to come first
fun(1, c=3, b=2)

0

---

## Operators for unpacking (`*` and  `**`)

In [9]:
my_list = [1, 2, 3]

In [10]:
my_list = [*my_list, 4]

In [11]:
my_list

[1, 2, 3, 4]

In [12]:
my_dict = {'a': 1, 'b': 2}

In [13]:
my_dict = {**my_dict, 'c': 3}

In [14]:
my_dict

{'a': 1, 'b': 2, 'c': 3}

---

## A function that takes any number of positional and keyword arguments

In [16]:
def takes_anything(*args, **kwargs):
    '''
    Positional arguments will be packaged into a tuple called 'args'.
    Keyword arguments will be packaged into a dictionary called 'kwargs'.
    '''
    print(f'Positional arguments: {args}')
    print(f'Keyword arguments:    {kwargs}')
    return

In [17]:
takes_anything(42, 1, 3, 'cat', dog='arf', a='b')

Positional arguments: (42, 1, 3, 'cat')
Keyword arguments:    {'dog': 'arf', 'a': 'b'}


---

## Pass a function to a function

In [22]:
def name_of_function(func: Callable) -> None:
    print(f'The name of the function is: {func.__name__}')
    return

In [23]:
name_of_function(fun)

The name of the function is: fun


In [24]:
def list_args_and_run(func: Callable, *args, **kwargs) -> Any:
    print(f'The name of the function is: {func.__name__}')
    print(f'Positional arguments: {args}')
    print(f'Keyword arguments:    {kwargs}')
    print(f'It was called at: {dt.now()}')
    return func(*args, **kwargs)

In [25]:
list_args_and_run(fun, 1, c=3, b=4)

The name of the function is: fun
Positional arguments: (1,)
Keyword arguments:    {'c': 3, 'b': 4}
It was called at: 2026-08-31 11:43:59.031036


2

---

## Decorators

A decorator is a function that takes a function as input and returns a modified version of the function.

In [2]:
def log_calls_simple(func: Callable) -> Callable:
    def wrapper(*args, **kwargs) -> Any:
        print(f'{func.__name__} was called: {dt.now()}')
        print(f'Positional arguments:       {args}')
        print(f'Keyword arguments:          {kwargs}')

        t0 = dt.now()
        result = func(*args, **kwargs)
        runtime = dt.now() - t0

        print(f'{func.__name__} ran for: {runtime}')
        return result
    return wrapper

In [10]:
@log_calls_simple
def wait(secs: float) -> str:
    time.sleep(secs)
    return f'I waited for {secs} secs'

In [11]:
wait(3)

wait was called: 2026-09-02 12:16:25.414413
Positional arguments:       (3,)
Keyword arguments:          {}
wait ran for: 0:00:03.001039


'I waited for 3 secs'

---

## Decorator Factory

This is a function that returns a decorator. For this implementation, it will allow us to build some options into the decorator.

In [33]:
def log_calls(show_args: bool = True,
              show_runtime: bool = True
             ) -> Callable:
    '''
    Used to show positional and keyword arguments and print runtime.
    '''
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args, **kwargs) -> Any:
            '''
            I wrap functions.
            '''
            print(f'{func.__name__} was called: {dt.now()}')

            if show_args:
                print(f'\tPositional arguments:       {args}')
                print(f'\tKeyword arguments:          {kwargs}')
    
            t0 = dt.now()
            result = func(*args, **kwargs)
            runtime = dt.now() - t0

            if show_runtime:
                print(f'{func.__name__} ran for: {runtime}\n')
            return result
        return wrapper
    return decorator

In [34]:
@log_calls()
def wait_again(secs: float) -> str:
    '''
    I wait for a specified number of seconds.
    '''
    time.sleep(secs)
    return f'I waited for {secs} secs'

@log_calls(show_args=False)
def big_input_func(x: str) -> int:
    return len(x)

In [24]:
def main():
    x = 'abc'*100
    print(wait_again(2))
    print(big_input_func(x))

In [25]:
main()

wait_again was called: 2026-09-02 12:32:24.095059
	Positional arguments:       (2,)
	Keyword arguments:          {}
wait_again ran for: 0:00:02.005066

I waited for 2 secs
big_input_func was called: 2026-09-02 12:32:26.100467
big_input_func ran for: 0:00:00.000004

300
